# Local SPFC dataset generation with Ollama

Generate AIM-Flow `spfc_decomposition_v1` files for both bundled 100-prompt manifests:

- GenEval: `configs/geneval_100_seed13.json`
- T2I-CompBench: `configs/t2i_compbench_100_seed13.json`

Everything runs locally through Ollama and the already-pulled `qwen3.6:27b` model. The notebook makes no internet requests, supports safe resume, validates every item, and writes new files under `outputs/ollama_datasets/`.

In [1]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

# Works when the kernel starts in either the repository root or notebooks/.
cwd = Path.cwd().resolve()
REPO = cwd if (cwd / "configs").is_dir() else cwd.parent
assert (REPO / "configs").is_dir(), f"Could not locate repository from {cwd}"

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3.6:27b"
OUTPUT_DIR = REPO / "outputs" / "ollama_datasets"
TEMPERATURE = 0.15
SEED = 13
# 32K comfortably fits the full decomposition guide, examples, schema, and response.
NUM_CTX = 32768
MAX_RETRIES = 3
RETRY_SECONDS = 2
REQUEST_TIMEOUT_SECONDS = 600

# Use None for all 100 items in each dataset; set a small integer for a smoke test.
LIMIT: int | None = None
# False resumes valid items already present in the output. True starts each output afresh.
OVERWRITE = False

DATASETS = {
    "geneval": REPO / "configs" / "geneval_100_seed13.json",
    "t2i_compbench": REPO / "configs" / "t2i_compbench_100_seed13.json",
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO)
print("Outputs:   ", OUTPUT_DIR)

Repository: /media/fezan/ASi/DVLM/steering/aim-flow
Outputs:    /media/fezan/ASi/DVLM/steering/aim-flow/outputs/ollama_datasets


## Verify the local runtime

Only loopback Ollama URLs are accepted. Start Ollama first (`ollama serve`) if the next cell cannot connect. The model must already be pulled; this notebook deliberately never pulls or downloads anything.

In [2]:
def local_ollama_request(path: str, payload: dict[str, Any] | None = None) -> dict[str, Any]:
    parsed = urlparse(OLLAMA_URL)
    if parsed.scheme != "http" or parsed.hostname not in {"127.0.0.1", "localhost", "::1"}:
        raise ValueError("OLLAMA_URL must be a local, plain-HTTP loopback address")
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        OLLAMA_URL.rstrip("/") + path,
        data=body,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            return json.loads(response.read().decode("utf-8"))
    except (HTTPError, URLError, TimeoutError) as exc:
        raise RuntimeError(f"Local Ollama request failed ({path}): {exc}") from exc

tags = local_ollama_request("/api/tags").get("models", [])
installed = sorted(model.get("name", "") for model in tags)
if MODEL not in installed:
    raise RuntimeError(f"Required local model {MODEL!r} is not installed. Installed: {installed}")
for name, path in DATASETS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
print("Ready — local model found:", MODEL)
print("Network target:", OLLAMA_URL, "(loopback only)")

Ready — local model found: qwen3.6:27b
Network target: http://127.0.0.1:11434 (loopback only)


In [3]:
ITEM_SCHEMA = {
    "type": "object",
    "properties": {
        "source_prompt": {"type": "string"},
        "negative_prompt": {"type": "string"},
        "primitive_prompts": {
            "type": "array",
            "minItems": 2,
            "maxItems": 8,
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "text": {"type": "string"},
                    "role": {"type": "string", "enum": ["primitive"]},
                    "weight": {"type": "number", "minimum": 0, "maximum": 2},
                    "enabled": {"type": "boolean"}
                },
                "required": ["name", "text", "role", "weight", "enabled"],
                "additionalProperties": False
            }
        }
    },
    "required": ["source_prompt", "negative_prompt", "primitive_prompts"],
    "additionalProperties": False
}

SYSTEM_PROMPT = """You create high-quality prompt decompositions for AIM-Flow/SPFC text-to-image generation.

Return only the JSON object required by the supplied schema. Do not include explanations, Markdown, or extra keys.

## Objective

Decompose the supplied target prompt into:

* A minimally ablated `source_prompt`
* Focused `primitives`
* A concise, target-specific `negative_prompt`

The decomposition should teach a diffusion model the individual entities, attribute bindings, counts, and relations before reconstructing the complete target.

## 1. Parse the target

Silently identify:

* **Entities:** objects, people, animals, surfaces, and containers
* **Attributes:** colour, material, shape, texture, size, clothing, and state
* **Bindings:** which attribute belongs to which entity
* **Counts:** exact number of each entity
* **Relations:** left/right, above/below, on top of/underneath, in front of/behind, inside/containing, beside, near, between, holding, wearing, reflected in, and similar relations
* **Global descriptors:** style, lighting, medium, camera, and quality terms

Preserve every explicit constraint. Do not swap attributes, reverse relations, or invent unstated details.

## 2. Source prompt

`source_prompt` must be a minimal, non-contradictory ablation of the target.

* Preserve the target’s entities and neutral scene context.
* Preserve relevant style, lighting, medium, and quality terms.
* Remove the attributes, counts, or relations that the primitives are intended to introduce.
* Do not replace removed details with contradictory details.
* For simple two-entity relation prompts, list both entities without asserting the tested relation.

Example:

Target:
`A television on the right of a woman.`

Source:
`A television and a woman.`

Do not introduce a substitute relation such as “near,” “beside,” or “together” unless it is explicitly present in the target.

## 3. Primitive construction

Use 2–8 primitives in total, including the final complete-target primitive.

Create primitives in this order when applicable:

1. Entity identities
2. Attribute-bound entities
3. Exact counts
4. Primary relations
5. Useful inverse relations
6. Complete target

Each primitive must be concise, standalone, visually grounded, and renderable.

### Entity primitives

Use separate identity primitives when they provide useful grounding.

Examples:

* `An airplane.`
* `A mouse.`

Do not add unnecessary identity primitives when the entity is already clearly established by a more informative attribute-bound primitive.

### Attribute primitives

Bind every attribute explicitly to its correct entity.

Correct:

* `A red cube.`
* `A woman wearing a blue shirt.`
* `A cat made of glass.`

Avoid:

* `A cube and red.`
* `A woman and a blue shirt.`
* `Glass and a cat.`

When multiple entities have attributes, create separate primitives when needed to prevent colour, material, or clothing swaps.

### Count primitives

Preserve exact counts explicitly.

Correct:

* `Exactly three yellow apples.`
* `Two dogs and one cat.`

Do not replace exact counts with vague plurals.

### Relation primitives

Every relation primitive must include:

* The subject
* The complete relation
* The reference entity or destination

Correct:

* `A television positioned to the right of a woman.`
* `An airplane positioned above a mouse.`
* `Three apples contained inside a bowl.`

Avoid:

* `A television on the right.`
* `An airplane and a mouse.`
* `Apples with a bowl.`

Preserve the relation direction exactly. Interpret left and right from the image viewer’s perspective unless the target explicitly specifies another viewpoint.

Do not strengthen the relation with words such as `directly`, `touching`, `attached`, `centered`, or `perfectly` unless the target requires them.

## 4. Inverse-relation policy

Add an inverse-relation primitive only when it provides a distinct directional description of the same scene.

Use inverse primitives for asymmetric relation pairs such as:

* `X left of Y` → `Y right of X`
* `X right of Y` → `Y left of X`
* `X above Y` → `Y below X`
* `X on top of Y` → `Y underneath X`
* `X in front of Y` → `Y behind X`
* `X inside Y` → `Y containing X`

Example:

* `An airplane positioned above a mouse.`
* `A mouse positioned underneath an airplane.`

Do not add inverse primitives for symmetric or nearly symmetric relations, because they provide duplicate supervision.

Do not invert:

* beside
* next to
* near
* adjacent to
* overlapping with

For example, these are duplicates and should not both appear:

* `A fish beside a bag.`
* `A bag beside a fish.`

Use only one clear relation primitive for symmetric relations.

## 5. Ambiguous or awkward target wording

The final primitive must preserve the target exactly, but focused primitives may use clearer grammatical wording.

Choose the most literal visually plausible interpretation without inventing extra meaning.

Example:

Target:
`a fish on side of a bag`

Acceptable relation primitive:
`A fish positioned at the side of a bag.`

Do not reinterpret it as:

* a fish inside the bag
* a fish printed on the bag
* a fish attached to the bag
* a fish on top of the bag

unless the target explicitly states that meaning.

When the relation remains ambiguous, preserve its general meaning rather than making it more specific.

## 6. Complex effects

Reflections, shadows, occlusions, and unusual interactions must remain complete visual events.

Correct:

* `A fox statue reflected in a cracked mirror.`
* `The shadow cast by a fox statue forms the shape of a dragon on the wall.`

Avoid:

* `A reflected fox.`
* `A dragon shadow.`
* `A dragon on the wall.`

Separate the underlying object, its special attribute, and the complex effect when each is an important target constraint.

## 7. Avoid redundant primitives

Every primitive must add a distinct supervision signal.

Do not create:

* Two paraphrases of the same symmetric relation
* A generic object primitive followed by an almost identical object primitive
* Both a relation and its inverse when the inverse adds no directional information
* Multiple primitives that differ only through minor wording

Prefer fewer strong primitives over redundant ones.

## 8. Final primitive

The final primitive must:

* Be an exact character-for-character copy of the supplied target prompt
* Appear last
* Have weight `1.1`
* Represent the complete target scene

Do not correct its spelling, grammar, capitalization, punctuation, or wording.

All preceding primitives should normally have weight `1.0`.

## 9. Naming and fixed fields

Primitive names must be unique snake_case identifiers prefixed sequentially:

* `S1_...`
* `S2_...`
* `S3_...`

Every primitive must have:

* `"role": "primitive"`
* `"enabled": true`

## 10. Negative prompt

Treat `negative_prompt` as a light diffusion-conditioning signal, not as a logical explanation.

Use a concise comma-separated list of approximately 6–14 visible failure modes.

Prioritize:

* Missing required entities
* Wrong entity identity
* Wrong material, colour, shape, or clothing
* Attributes attached to the wrong entity
* Incorrect count
* One or two likely incorrect spatial configurations
* Extra salient objects
* Blur, distortion, malformed objects
* Low resolution
* Text, logos, watermarks

Describe concrete unwanted images rather than abstract reasoning.

Prefer:

* `airplane below mouse`
* `mouse above airplane`
* `red sphere`
* `blue cube`
* `fish inside bag`
* `four apples`

Avoid:

* `incorrect spatial relation`
* `reversed relation`
* `wrong composition`
* `object not correctly positioned`
* long logical sentences
* many paraphrases of the same failure

Do not include every possible wrong relation. Include only the most likely confusions.

Never negate anything required by the target.

## Examples

### Example 1: Asymmetric spatial relation

Target:
`an airplane on top of a mouse`

Output:
{
"source_prompt": "An airplane and a mouse.",
"primitive_prompts": [
{
"name": "S1_airplane",
"text": "An airplane.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S2_mouse",
"text": "A mouse.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S3_airplane_above_mouse",
"text": "An airplane positioned above a mouse.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S4_mouse_underneath_airplane",
"text": "A mouse positioned underneath an airplane.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S5_complete_target",
"text": "an airplane on top of a mouse",
"weight": 1.1,
"role": "primitive",
"enabled": true
}
],
"negative_prompt": "missing airplane, missing mouse, airplane below mouse, mouse above airplane, extra salient objects, malformed airplane, malformed mouse, blur, low resolution, text, watermarks"
}

### Example 2: Symmetric relation

Target:
`a fish on side of a bag`

Output:
{
"source_prompt": "A fish and a bag.",
"primitive_prompts": [
{
"name": "S1_fish",
"text": "A fish.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S2_bag",
"text": "A bag.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S3_fish_at_side_of_bag",
"text": "A fish positioned at the side of a bag.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S4_complete_target",
"text": "a fish on side of a bag",
"weight": 1.1,
"role": "primitive",
"enabled": true
}
],
"negative_prompt": "missing fish, missing bag, fish inside bag, fish above bag, fish underneath bag, extra salient objects, malformed fish, malformed bag, blur, low resolution, text, watermarks"
}

### Example 3: Colour binding and direction

Target:
`A red cube to the left of a blue sphere.`

Output:
{
"source_prompt": "A cube and a sphere.",
"primitive_prompts": [
{
"name": "S1_red_cube",
"text": "A red cube.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S2_blue_sphere",
"text": "A blue sphere.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S3_red_cube_left_of_blue_sphere",
"text": "A red cube positioned to the left of a blue sphere.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S4_blue_sphere_right_of_red_cube",
"text": "A blue sphere positioned to the right of a red cube.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S5_complete_target",
"text": "A red cube to the left of a blue sphere.",
"weight": 1.1,
"role": "primitive",
"enabled": true
}
],
"negative_prompt": "missing cube, missing sphere, blue cube, red sphere, cube right of sphere, sphere left of cube, extra salient objects, distorted shapes, blur, low resolution, text, watermarks"
}

### Example 4: Count and containment

Target:
`Three yellow apples inside a blue bowl.`

Output:
{
"source_prompt": "Apples and a bowl.",
"primitive_prompts": [
{
"name": "S1_three_yellow_apples",
"text": "Exactly three yellow apples.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S2_blue_bowl",
"text": "A blue bowl.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S3_apples_inside_bowl",
"text": "Exactly three yellow apples contained inside a blue bowl.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S4_bowl_containing_apples",
"text": "A blue bowl containing exactly three yellow apples.",
"weight": 1.0,
"role": "primitive",
"enabled": true
},
{
"name": "S5_complete_target",
"text": "Three yellow apples inside a blue bowl.",
"weight": 1.1,
"role": "primitive",
"enabled": true
}
],
"negative_prompt": "missing apples, missing bowl, two apples, four apples, non-yellow apples, non-blue bowl, apples outside bowl, apples beside bowl, extra fruit, blur, low resolution, text, watermarks"
}

Before returning the JSON, silently verify:

* Every required entity is represented.
* Every attribute is bound to the correct entity.
* Exact counts are preserved.
* Relations retain the correct subject, reference, and direction.
* Inverse relations are used only for asymmetric relations.
* Symmetric relations are not duplicated.
* Ambiguous wording has not been overinterpreted.
* Every primitive contributes distinct supervision.
* Negative phrases describe concrete visible failures.
* The source does not contradict the target.
* The final primitive exactly matches the supplied target.
"""

def user_prompt(sample: dict[str, Any], feedback: str | None = None) -> str:
    text = (
        f"Dataset: {sample['source']}\n"
        f"Category: {sample['category']}\n"
        f"Target prompt: {sample['prompt']}\n"
        f"Metadata: {json.dumps(sample.get('metadata', {}), sort_keys=True)}\n"
        "Create its SPFC decomposition now."
    )
    return text if not feedback else text + f"\nYour previous result was invalid: {feedback}\nCorrect it."

def parse_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I | re.S).strip()
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("model response is not a JSON object")
    return value

def validate_generated(sample: dict[str, Any], generated: dict[str, Any]) -> dict[str, Any]:
    allowed = {"source_prompt", "negative_prompt", "primitive_prompts"}
    if set(generated) != allowed:
        raise ValueError(f"expected keys {sorted(allowed)}, got {sorted(generated)}")
    source = str(generated["source_prompt"]).strip()
    negative = str(generated["negative_prompt"]).strip()
    primitives = generated["primitive_prompts"]
    if not source or not negative:
        raise ValueError("source_prompt and negative_prompt must be non-empty")
    if not isinstance(primitives, list) or not 2 <= len(primitives) <= 8:
        raise ValueError("primitive_prompts must contain 2-8 items")
    names = []
    for index, primitive in enumerate(primitives, 1):
        if set(primitive) != {"name", "text", "role", "weight", "enabled"}:
            raise ValueError(f"primitive {index} has unexpected fields")
        name, text = str(primitive["name"]).strip(), str(primitive["text"]).strip()
        if not re.fullmatch(rf"S{index}_[a-z0-9_]+", name):
            raise ValueError(f"primitive {index} name must match S{index}_snake_case")
        if not text or primitive["role"] != "primitive" or primitive["enabled"] is not True:
            raise ValueError(f"primitive {index} has invalid text, role, or enabled value")
        weight = float(primitive["weight"])
        if not 0 <= weight <= 2:
            raise ValueError(f"primitive {index} weight is outside [0, 2]")
        primitive.update(name=name, text=text, weight=weight)
        names.append(name)
    if len(names) != len(set(names)):
        raise ValueError("primitive names must be unique")
    if primitives[-1]["text"] != sample["prompt"]:
        raise ValueError("final primitive text must exactly match the target prompt")
    if abs(float(primitives[-1]["weight"]) - 1.1) > 1e-9:
        raise ValueError("final primitive weight must be 1.1")
    return {
        "id": sample["id"],
        "target_prompt": sample["prompt"].strip(),
        "source_prompt": source,
        "negative_prompt": negative,
        "primitive_prompts": primitives,
        "metadata": {
            **sample.get("metadata", {}),
            "category": sample["category"],
            "template": False,
            "generator": "ollama",
            "model": MODEL,
        },
    }

def generate_item(sample: dict[str, Any]) -> dict[str, Any]:
    feedback = None
    for attempt in range(1, MAX_RETRIES + 1):
        payload = {
            "model": MODEL,
            "stream": False,
            "think": False,
            "format": ITEM_SCHEMA,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt(sample, feedback)},
            ],
            "options": {"temperature": TEMPERATURE, "seed": SEED, "num_ctx": NUM_CTX},
            "keep_alive": "30m",
        }
        try:
            response = local_ollama_request("/api/chat", payload)
            generated = parse_json_object(response["message"]["content"])
            return validate_generated(sample, generated)
        except Exception as exc:
            feedback = f"{type(exc).__name__}: {exc}"
            if attempt == MAX_RETRIES:
                raise RuntimeError(f"Failed {sample['id']} after {MAX_RETRIES} attempts: {feedback}") from exc
            print(f"  retry {attempt}/{MAX_RETRIES - 1}: {feedback}")
            time.sleep(RETRY_SECONDS)
    raise AssertionError("unreachable")

print("Generation and validation helpers loaded.")

Generation and validation helpers loaded.


In [ ]:
def atomic_save(data: dict[str, Any], path: Path) -> None:
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temp, path)

def load_manifest(path: Path) -> dict[str, Any]:
    manifest = json.loads(path.read_text(encoding="utf-8"))
    samples = manifest.get("samples")
    if not isinstance(samples, list) or not samples:
        raise ValueError(f"No samples in {path}")
    ids = [sample.get("id") for sample in samples]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate sample IDs in {path}")
    return manifest

def load_resumable_items(output_path: Path, samples_by_id: dict[str, dict[str, Any]]) -> dict[str, dict[str, Any]]:
    if OVERWRITE or not output_path.exists():
        return {}
    saved = json.loads(output_path.read_text(encoding="utf-8"))
    if saved.get("schema_version") != "spfc_decomposition_v1":
        raise ValueError(f"Cannot resume unsupported schema in {output_path}")
    valid = {}
    for item in saved.get("items", []):
        sample = samples_by_id.get(item.get("id"))
        if sample is None:
            continue
        generated = {key: item[key] for key in ("source_prompt", "negative_prompt", "primitive_prompts")}
        valid[item["id"]] = validate_generated(sample, generated)
    return valid

def generate_dataset(name: str, manifest_path: Path) -> Path:
    manifest = load_manifest(manifest_path)
    samples = manifest["samples"][:LIMIT] if LIMIT is not None else manifest["samples"]
    samples_by_id = {sample["id"]: sample for sample in samples}
    suffix = f"_{LIMIT}" if LIMIT is not None else ""
    output_path = OUTPUT_DIR / f"{name}{suffix}_seed{manifest['seed']}_spfc_ollama.json"
    completed = load_resumable_items(output_path, samples_by_id)
    print(f"\n{name}: {len(samples)} samples; resuming {len(completed)} valid items")

    started = time.monotonic()
    for index, sample in enumerate(samples, 1):
        if sample["id"] in completed:
            print(f"[{index:03d}/{len(samples):03d}] {sample['id']} (cached)")
            continue
        print(f"[{index:03d}/{len(samples):03d}] {sample['id']} — {sample['prompt']}")
        completed[sample["id"]] = generate_item(sample)
        ordered = [completed[s["id"]] for s in samples if s["id"] in completed]
        atomic_save({"schema_version": "spfc_decomposition_v1", "items": ordered}, output_path)
    elapsed = time.monotonic() - started
    print(f"Completed {name} in {elapsed / 60:.1f} min → {output_path}")
    return output_path

# Both datasets are generated sequentially to avoid loading a second 17 GB model context.
generated_paths = [generate_dataset(name, path) for name, path in DATASETS.items()]
generated_paths


geneval: 100 samples; resuming 19 valid items
[001/100] geneval_color_000000 (cached)
[002/100] geneval_color_000001 (cached)
[003/100] geneval_color_000002 (cached)
[004/100] geneval_color_000003 (cached)
[005/100] geneval_color_000004 (cached)
[006/100] geneval_color_000005 (cached)
[007/100] geneval_color_000006 (cached)
[008/100] geneval_color_000007 (cached)
[009/100] geneval_color_000008 (cached)
[010/100] geneval_color_000009 (cached)
[011/100] geneval_color_000010 (cached)
[012/100] geneval_color_000011 (cached)
[013/100] geneval_color_000012 (cached)
[014/100] geneval_color_000013 (cached)
[015/100] geneval_color_000014 (cached)
[016/100] geneval_color_000015 (cached)
[017/100] geneval_color_000016 (cached)
[018/100] geneval_color_000017 (cached)
[019/100] geneval_color_000018 (cached)
[020/100] geneval_color_000019 — A photo of a white toilet and a black computer mouse


## Final repository-schema validation

This imports AIM-Flow's own schema and verifies that every output can be consumed by the generation pipeline.

In [ ]:
import sys

src_path = str(REPO / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from aim_flow.eval_bench.schemas import DecompositionManifest

for path in generated_paths:
    result = DecompositionManifest.load(path)
    print(f"PASS: {path.name}: {len(result.items)} valid decomposition items")